# 🚇 NYC Transit Real-Time Passenger Flow & Overcrowding Analytics
## Exploratory Data Analysis, Entity Resolution, K-Means Clustering & Wait Assessment Correlation

**Học phần:** IT4931 - Lưu trữ và Xử lý Dữ liệu lớn (SoICT - HUST)  
Notebook này thực hiện khảo sát thực nghiệm trên tập dữ liệu thật trong `data/archive/`, kiểm chứng 5 thách thức kỹ thuật cốt lõi và trực quan hóa kết quả từ **Batch Layer** & **Speed Layer**.

In [ ]:
import os
import sys
import glob
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.batch.entity_resolution import StationEntityResolver
from src.batch.clean_turnstile import run_local_fallback_job
from src.batch.kmeans_clustering import run_baseline_kmeans_fallback
from src.batch.correlation_analysis import run_correlation_analysis
from src.streaming.stream_processor import run_local_sliding_window_simulation

plt.style.use('seaborn-v0_8-whitegrid')
print('Project Root:', PROJECT_ROOT)

## 1. Khảo sát Dữ liệu Nguồn & Kiểm chứng Entity Resolution (> 99% Match)
Kiểm tra bảng ánh xạ `StationEntityResolver` giữa tên trạm viết tắt trong `TurnstileData` (`GRD CNTRL-42 ST`, `34 ST-PENN STA`, `42 ST-PORT AUTH`) và danh mục không gian chuẩn `MTA_StationLocations.csv`.

In [ ]:
resolver = StationEntityResolver(os.path.join(PROJECT_ROOT, 'data/archive/MTA_StationLocations.csv'))
sample_stations = [
    ('GRD CNTRL-42 ST', '4567S', 'IRT'),
    ('34 ST-PENN STA', '123ACE', 'IRT'),
    ('42 ST-PORT AUTH', 'ACENQRS1237', 'IND'),
    ('47-50 STS ROCK', 'BDFM', 'IND'),
    ('ATL AV-BARCLAY', 'BDNQR2345', 'BMT'),
    ('JKSN HT-ROOSVLT', 'EFMR7', 'IND'),
    ('EXCHANGE PLACE', '1', 'PTH'),  # Non-MTA PATH station -> Filtered out
]
resolved_demos = []
for st, ln, div in sample_stations:
    res = resolver.resolve(st, ln, div)
    resolved_demos.append({
        'Raw Turnstile Station': st,
        'Division': div,
        'Matched MTA Stop Name': res['stop_name'] if res else 'FILTERED (Non-MTA)',
        'Station ID': res['station_id'] if res else None,
        'Borough': res['borough'] if res else None,
        'Coordinates': f"({res['latitude']}, {res['longitude']})" if res else None
    })
pd.DataFrame(resolved_demos)

## 2. Làm sạch Bộ đếm Lũy kế (Counter Rollover & Spike Filtering) & Tính Baseline

In [ ]:
clean_summary = run_local_fallback_job(
    input_pattern=os.path.join(PROJECT_ROOT, 'data/archive/TurnstileData/turnstile_*.txt'),
    locations_csv=os.path.join(PROJECT_ROOT, 'data/archive/MTA_StationLocations.csv'),
    output_parquet=os.path.join(PROJECT_ROOT, 'data/processed/turnstile_parquet'),
    baseline_output=os.path.join(PROJECT_ROOT, 'data/processed/station_baseline.json'),
    max_files=4
)
pd.Series(clean_summary, name='Batch Cleaning Metrics')

## 3. Phân cụm Hành vi Nhà ga bằng K-Means (24-Hour Temporal Profile)

In [ ]:
kmeans_summary = run_baseline_kmeans_fallback(
    baseline_json=os.path.join(PROJECT_ROOT, 'data/processed/station_baseline.json'),
    output_json=os.path.join(PROJECT_ROOT, 'data/processed/station_clusters.json'),
    k=3
)
with open(os.path.join(PROJECT_ROOT, 'data/processed/station_clusters.json'), encoding='utf-8') as f:
    clusters_data = json.load(f)

clusters_df = pd.DataFrame(clusters_data)
print(f"Silhouette Score: {kmeans_summary['silhouette_score']}")
clusters_df[['station_id', 'stop_name', 'borough', 'archetype', 'estimated_daily_flow']].head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for archetype, grp in clusters_df.groupby('archetype'):
    mean_curve = np.mean(np.vstack(grp['hourly_profile_24h'].values), axis=0)
    ax.plot(range(24), mean_curve * 100, marker='o', linewidth=2, label=f"{archetype} (n={len(grp)})")

ax.set_title('24-Hour Passenger Flow Temporal Profiles by K-Means Station Archetype', fontsize=13)
ax.set_xlabel('Hour of Day (00:00 - 23:00)')
ax.set_ylabel('Share of Daily Station Traffic (%)')
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()

## 4. Phân tích Tương quan Quá tải Nhà ga với Chỉ số Trễ tàu (`Subway Wait Assessment`)

In [ ]:
corr_res = run_correlation_analysis(
    turnstile_pattern=os.path.join(PROJECT_ROOT, 'data/archive/TurnstileData/turnstile_*.txt'),
    perf_csv_path=os.path.join(PROJECT_ROOT, 'data/archive/NYC_TransitPerformance/MTA_Performance_Agencies.csv'),
    output_json=os.path.join(PROJECT_ROOT, 'data/processed/correlation_results.json'),
    max_files=6
)
pd.DataFrame(corr_res['line_summaries']).head(10)

## 5. Kiểm thử Speed Layer: Cửa sổ trượt 15 phút & Cảnh báo Quá tải Z-Score

In [ ]:
stream_res = run_local_sliding_window_simulation(
    turnstile_pattern=os.path.join(PROJECT_ROOT, 'data/archive/TurnstileData/turnstile_*.txt'),
    locations_csv=os.path.join(PROJECT_ROOT, 'data/archive/MTA_StationLocations.csv'),
    baseline_json=os.path.join(PROJECT_ROOT, 'data/processed/station_baseline.json'),
    output_json=os.path.join(PROJECT_ROOT, 'data/processed/realtime_alerts.json'),
    max_events=25000
)
pd.DataFrame(stream_res['top_5_emergency_stations'])[
    ['station_id', 'stop_name', 'borough', 'window_end', 'flow_rate_15m', 'baseline_mean_15m', 'z_score', 'alert_status']
]